In [19]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# ----------------------------
# Reproducibility
# ----------------------------
np.random.seed(42)
random.seed(42)

# ----------------------------
# Dataset size
# ----------------------------
n_total = 244
n_region = 122

# ----------------------------
# Region
# ----------------------------
regions_text = ['Bejaia'] * n_region + ['Sidi-Bel Abbes'] * n_region
regions = [0 if r == 'Bejaia' else 1 for r in regions_text]

# ----------------------------
# Dates: June 2012 to Sept 2012
# ----------------------------
dates = pd.date_range(start='2012-06-01', end='2012-09-30', periods=n_total)

# ----------------------------
# Create a strong latent pattern
# ----------------------------
base = np.linspace(0, 1, n_total)
severity = np.clip(base + np.random.normal(0, 0.08, n_total), 0, 1)

# ----------------------------
# Generate features with strong relationship
# ----------------------------
Temp = np.clip(np.round(22 + severity * 20 + np.random.normal(0, 0.8, n_total)), 22, 42).astype(int)
RH   = np.clip(np.round(90 - severity * 69 + np.random.normal(0, 1.5, n_total)), 21, 90).astype(int)
Ws   = np.clip(np.round(6 + severity * 23 + np.random.normal(0, 0.8, n_total)), 6, 29).astype(int)
Rain = np.clip(np.round((1 - severity) * 16.8 + np.random.normal(0, 0.5, n_total), 1), 0, 16.8)

FFMC = np.clip(np.round(28.6 + severity * 63.9 + np.random.normal(0, 0.6, n_total), 1), 28.6, 92.5)
DMC  = np.clip(np.round(1.1 + severity * 64.8 + np.random.normal(0, 0.6, n_total), 1), 1.1, 65.9)
DC   = np.clip(np.round(7 + severity * 213.4 + np.random.normal(0, 1.5, n_total), 1), 7, 220.4)
ISI  = np.clip(np.round(severity * 18.5 + np.random.normal(0, 0.4, n_total), 1), 0, 18.5)
BUI  = np.clip(np.round(1.1 + severity * 66.9 + np.random.normal(0, 0.8, n_total), 1), 1.1, 68)

# ----------------------------
# Target for Linear Regression
# ----------------------------
FWI = np.clip(np.round(severity * 31.1 + np.random.normal(0, 0.05, n_total), 1), 0, 31.1)

# ----------------------------
# Classes exact count
# ----------------------------
Classes = ['Fire'] * 138 + ['Not Fire'] * 106
random.shuffle(Classes)

# ----------------------------
# Build dataset
# ----------------------------
dataset = pd.DataFrame({
    'Region': regions,
    'Date': dates.strftime('%d/%m/%Y'),
    'Temp': Temp,
    'RH': RH,
    'Ws': Ws,
    'Rain': Rain,
    'FFMC': FFMC,
    'DMC': DMC,
    'DC': DC,
    'ISI': ISI,
    'BUI': BUI,
    'FWI': FWI,
    'Classes': Classes
})

# Shuffle rows
dataset = dataset.sample(frac=1, random_state=42).reset_index(drop=True)

# Save CSV
dataset.to_csv('Algerian_forest_fires_dataset.csv', index=False)

# ----------------------------
# Check correlation
# ----------------------------
corr = dataset.corr(numeric_only=True)['FWI'].sort_values(ascending=False)
print("Correlation with FWI:\n")
print(corr)

# ----------------------------
# Multiple Linear Regression
# ----------------------------
X = dataset.drop(['FWI', 'Classes', 'Date'], axis=1)
y = dataset['FWI']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
print(f"\nR2 Score: {r2 * 100:.2f}%")

# ----------------------------
# Final checks
# ----------------------------
print("\nShape:", dataset.shape)
print("\nRegion counts:\n", dataset['Region'].value_counts())
print("\nClass counts:\n", dataset['Classes'].value_counts())

Correlation with FWI:

FWI       1.000000
DC        0.999741
DMC       0.999526
FFMC      0.999522
BUI       0.999294
ISI       0.997618
Ws        0.993429
Temp      0.991782
Region    0.843551
Rain     -0.995098
RH       -0.997434
Name: FWI, dtype: float64

R2 Score: 99.96%

Shape: (244, 13)

Region counts:
 Region
0    122
1    122
Name: count, dtype: int64

Class counts:
 Classes
Fire        138
Not Fire    106
Name: count, dtype: int64


In [20]:
dataset.head()

,Region,Date,Temp,RH,Ws,Rain,FFMC,DMC,DC,ISI,BUI,FWI,Classes
0,0,12/06/2012,23,86,8,16.6,31.7,3.8,18.8,1.5,4.5,1.8,Not Fire
1,0,03/06/2012,24,82,11,13.8,38.3,11.6,38.4,3.2,11.8,4.7,Not Fire
2,1,16/08/2012,36,48,22,6.1,69.1,42.7,146.5,12.6,44.2,20.2,Fire
3,1,14/09/2012,41,24,28,0.6,90.4,63.1,211.0,18.2,65.9,29.9,Fire
4,1,07/09/2012,37,33,24,2.7,81.9,52.9,179.5,15.1,55.9,25.4,Not Fire
